<a href="https://colab.research.google.com/github/Ankit0974/Plant-Disease-Detection-CNN-/blob/main/PlantDiseaseProjectfinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.0/231.0 kB 17.6 MB/s eta 0:00:00


In [ ]:
import os

os.environ["KAGGLE_API_TOKEN"] = "KGAT_6dacc7c207acfb2d686a16c8ac541339"

In [ ]:
!kaggle datasets list -s plantdisease

ref                                       title                                              size  lastUpdated                 downloadCount  voteCount  usabilityRating  
----------------------------------------  -------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
mummidi31/plantdisease                    Plantdisease                                      12215  2020-12-14 06:53:29.700000             39          4  0.1875           
ly9802/plantdisease                       PlantDisease                                  855079835  2021-12-10 05:52:35.963000             66          1  0.23529412       
minhaz027/plantdisease                    Plantdisease                                  153905669  2021-04-25 20:53:24.067000             32          2  0.0              
sayanroy058/plantdisease                  PlantDisease                                  934938040  2024-07-23 18:41:45.293000              5     

In [ ]:
!kaggle datasets download -d emmarex/plantdisease

Dataset URL: https://www.kaggle.com/datasets/emmarex/plantdisease
License(s): unknown
100% 658M/658M [00:05<00:00, 129MB/s]



In [ ]:
!unzip -q plantdisease.zip

In [ ]:
import os

dataset_path = "PlantVillage"  # adjust if needed

for cls in os.listdir(dataset_path):
    cls_path = os.path.join(dataset_path, cls)
    if os.path.isdir(cls_path):
        print(cls, len(os.listdir(cls_path)))

Pepper__bell___healthy 1478
Potato___Late_blight 1000
Tomato_Bacterial_spot 2127
Tomato_healthy 1591
Tomato__Tomato_YellowLeaf__Curl_Virus 3209
Tomato__Tomato_mosaic_virus 373
Potato___healthy 152
Pepper__bell___Bacterial_spot 997
Potato___Early_blight 1000
Tomato_Late_blight 1909
Tomato_Leaf_Mold 952
Tomato_Septoria_leaf_spot 1771
Tomato_Early_blight 1000
Tomato__Target_Spot 1404
Tomato_Spider_mites_Two_spotted_spider_mite 1676


In [ ]:
!pip install split-folders

In [ ]:
import splitfolders

splitfolders.ratio(
    "PlantVillage",
    output="dataset_split",
    seed=42,
    ratio=(0.7, 0.15, 0.15)
)

Copying files: 20639 files [00:03, 6262.10 files/s]


In [ ]:
import torch
import torchvision
import torch.nn as nn

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [ ]:
import torch
import torch.nn as nn
from torchvision import models

In [ ]:
model = torchvision.models.vgg16(weights="IMAGENET1K_V1")

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 190MB/s]


In [ ]:
model

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print(device)
model = model.to(device)

cuda


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [ ]:
val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [ ]:
train_dataset = datasets.ImageFolder(
    "dataset_split/train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    "dataset_split/val",
    transform=val_transform
)

test_dataset = datasets.ImageFolder(
    "dataset_split/test",
    transform=val_transform
)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

In [ ]:
pip install torchinfo

In [ ]:
from torchinfo import summary

summary(model, input_size=(32, 3, 224, 224))

Layer (type:depth-idx)                   Output Shape              Param #
VGG                                      [32, 1000]                --
├─Sequential: 1-1                        [32, 512, 7, 7]           --
│    └─Conv2d: 2-1                       [32, 64, 224, 224]        1,792
│    └─ReLU: 2-2                         [32, 64, 224, 224]        --
│    └─Conv2d: 2-3                       [32, 64, 224, 224]        36,928
│    └─ReLU: 2-4                         [32, 64, 224, 224]        --
│    └─MaxPool2d: 2-5                    [32, 64, 112, 112]        --
│    └─Conv2d: 2-6                       [32, 128, 112, 112]       73,856
│    └─ReLU: 2-7                         [32, 128, 112, 112]       --
│    └─Conv2d: 2-8                       [32, 128, 112, 112]       147,584
│    └─ReLU: 2-9                         [32, 128, 112, 112]       --
│    └─MaxPool2d: 2-10                   [32, 128, 56, 56]         --
│    └─Conv2d: 2-11                      [32, 256, 56, 56]         29

In [ ]:
model = models.vgg16(weights="DEFAULT")

    # Freeze everything
for param in model.features.parameters():
    param.requires_grad = False

    # Unfreeze selected blocks
blocks = [
    model.features[:5],
    model.features[5:10],
    model.features[10:17],
    model.features[17:24],
    model.features[24:]
    ]

for block in blocks[-1:]:
    for param in block.parameters():
        param.requires_grad = True

In [ ]:
num_classes = 15


In [ ]:
model

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [ ]:
layers = []
in_features = 25088

for _ in range(3):
    layers.extend([
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.BatchNorm1d(256),
        nn.Dropout(0.5)
    ])
    in_features = 256

layers.append(
    nn.Linear(256, 15)
)

model.classifier = nn.Sequential(*layers)
print(model.classifier)

model = model.to(device)

criterion = nn.CrossEntropyLoss()

Sequential(
  (0): Linear(in_features=25088, out_features=256, bias=True)
  (1): ReLU()
  (2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (3): Dropout(p=0.5, inplace=False)
  (4): Linear(in_features=256, out_features=256, bias=True)
  (5): ReLU()
  (6): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (7): Dropout(p=0.5, inplace=False)
  (8): Linear(in_features=256, out_features=256, bias=True)
  (9): ReLU()
  (10): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (11): Dropout(p=0.5, inplace=False)
  (12): Linear(in_features=256, out_features=15, bias=True)
)


In [ ]:
from torchinfo import summary
summary(model, input_size=(32, 3, 224, 224))

Layer (type:depth-idx)                   Output Shape              Param #
VGG                                      [32, 15]                  --
├─Sequential: 1-1                        [32, 512, 7, 7]           --
│    └─Conv2d: 2-1                       [32, 64, 224, 224]        (1,792)
│    └─ReLU: 2-2                         [32, 64, 224, 224]        --
│    └─Conv2d: 2-3                       [32, 64, 224, 224]        (36,928)
│    └─ReLU: 2-4                         [32, 64, 224, 224]        --
│    └─MaxPool2d: 2-5                    [32, 64, 112, 112]        --
│    └─Conv2d: 2-6                       [32, 128, 112, 112]       (73,856)
│    └─ReLU: 2-7                         [32, 128, 112, 112]       --
│    └─Conv2d: 2-8                       [32, 128, 112, 112]       (147,584)
│    └─ReLU: 2-9                         [32, 128, 112, 112]       --
│    └─MaxPool2d: 2-10                   [32, 128, 56, 56]         --
│    └─Conv2d: 2-11                      [32, 256, 56, 56]   

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

In [ ]:


model = model.to(device)

In [ ]:
print(type(model))

<class 'torchvision.models.vgg.VGG'>


In [ ]:
print(next(model.parameters()).device)

cuda:0


In [ ]:
print(model)

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [ ]:
print(optimizer)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0.0001
)


In [ ]:
for images, labels in train_loader:

    images = images.to(device)

    outputs = model(images)

    print(outputs.shape)

    break

torch.Size([16, 15])


In [ ]:
trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(trainable)

13639183


In [ ]:
print(model.classifier)

Sequential(
  (0): Linear(in_features=25088, out_features=256, bias=True)
  (1): ReLU()
  (2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (3): Dropout(p=0.5, inplace=False)
  (4): Linear(in_features=256, out_features=256, bias=True)
  (5): ReLU()
  (6): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (7): Dropout(p=0.5, inplace=False)
  (8): Linear(in_features=256, out_features=256, bias=True)
  (9): ReLU()
  (10): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (11): Dropout(p=0.5, inplace=False)
  (12): Linear(in_features=256, out_features=15, bias=True)
)


In [ ]:
print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

14440
3089
3109


In [ ]:

for epoch in range(15):

    # =====================
    # TRAINING
    # =====================
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()


        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = 100 * correct / total

    print(
        f"Epoch [{epoch+1}/15] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc:.2f}% | "

    )


Epoch [1/15] Train Loss: 1.5001 Train Acc: 54.91% | 
Epoch [2/15] Train Loss: 0.5992 Train Acc: 83.13% | 
Epoch [3/15] Train Loss: 0.3619 Train Acc: 89.47% | 
Epoch [4/15] Train Loss: 0.2678 Train Acc: 92.48% | 
Epoch [5/15] Train Loss: 0.2300 Train Acc: 93.35% | 
Epoch [6/15] Train Loss: 0.1791 Train Acc: 94.73% | 
Epoch [7/15] Train Loss: 0.2980 Train Acc: 90.91% | 
Epoch [8/15] Train Loss: 0.1668 Train Acc: 95.02% | 
Epoch [9/15] Train Loss: 0.1334 Train Acc: 96.10% | 
Epoch [10/15] Train Loss: 0.1066 Train Acc: 96.96% | 
Epoch [11/15] Train Loss: 0.1259 Train Acc: 96.18% | 
Epoch [12/15] Train Loss: 0.0940 Train Acc: 97.47% | 
Epoch [13/15] Train Loss: 0.0923 Train Acc: 97.32% | 
Epoch [14/15] Train Loss: 0.0841 Train Acc: 97.60% | 
Epoch [15/15] Train Loss: 0.0794 Train Acc: 97.60% | 


In [ ]:
# The validation accuracy of Resnet - 95.92%
# The validation accuracy of VGG16 - 94.33%
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_acc = 100 * correct / total

print(f"Test Accuracy: {test_acc:.2f}%")

Test Accuracy: 97.78%


In [ ]:
# After training finishes
torch.save(model.state_dict(), "vgg16_plant_disease.pth")

print("Model saved successfully!")

Model saved successfully!


In [ ]:
model.eval()

image, label = test_dataset[900]

with torch.no_grad():
    output = model(image.unsqueeze(0).to(device))
    pred = output.argmax(1)

print("Actual:", test_dataset.classes[label])
print("Predicted:", test_dataset.classes[pred.item()])

Actual: Tomato_Bacterial_spot
Predicted: Tomato_Bacterial_spot


In [ ]:
correct = 0

for i in range(100):
    image, label = test_dataset[i]

    with torch.no_grad():
        output = model(image.unsqueeze(0).to(device))
        pred = output.argmax(1).item()

    if pred == label:
        correct += 1

print("Correct:", correct)
print("Accuracy:", correct/100)

Correct: 96
Accuracy: 0.96


In [ ]:
for i in range(10):

    image, label = test_dataset[i]

    with torch.no_grad():
        output = model(image.unsqueeze(0).to(device))
        pred = output.argmax(1).item()

    print(
        f"Actual: {label}  Pred: {pred}"
    )

Actual: 0  Pred: 0
Actual: 0  Pred: 0
Actual: 0  Pred: 0
Actual: 0  Pred: 0
Actual: 0  Pred: 0
Actual: 0  Pred: 0
Actual: 0  Pred: 0
Actual: 0  Pred: 0
Actual: 0  Pred: 0
Actual: 0  Pred: 0
